# Gamil RAG project

### Step 1: Import library

In [47]:
import os
import ollama
import json
from dotenv import load_dotenv
from huggingface_hub import login
from sentence_transformers import SentenceTransformer
import gradio as gr
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
import base64
from datetime import datetime

### Step 2: setting variables

In [48]:
load_dotenv(override = True)
ollama_api_key = os.getenv("OLLAMA_API_KEY")
hf_token = os.getenv("HF_TOKEN")
token_path = r'.credentials\token.json'
credential_path = r'.credentials\credentials.json'
login(hf_token, add_to_git_credential=True)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### Step3: Access gmail

In [42]:
# If modifying these scopes, delete the file token.json.
SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']

def login_gmail(credential_path, token_path = None):
    creds = None
    # token.json stores the user's access and refresh tokens
    if os.path.exists(token_path):
        creds = Credentials.from_authorized_user_file(token_path, SCOPES)
    
    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(credential_path, SCOPES)
            creds = flow.run_local_server(port=0)
        with open(token_path, 'w') as token:
            token.write(creds.to_json())

    service = build('gmail', 'v1', credentials=creds)
    return service

def get_simplified_message(service, msg_id):
    # Fetch message with 'full' format to get headers and body
    message = service.users().messages().get(userId='me', id=msg_id, format='full').execute()
    
    payload = message.get('payload', {})
    headers = payload.get('headers', [])
    
    # 1. Extract Title (Subject) and Sender (From)
    title = "No Subject"
    sender = "Unknown Sender"
    
    for h in headers:
        if h['name'] == 'Subject':
            title = h['value']
        if h['name'] == 'From':
            sender = h['value']
    
    # 2. Extract and format Date/Time
    # internalDate is in milliseconds (UTC)
    timestamp = int(message.get('internalDate')) / 1000
    dt_object = datetime.fromtimestamp(timestamp)
    formatted_date = dt_object.strftime('%Y-%m-%d %H:%M:%S')

    # 3. Extract Body (Plain Text only, ignoring images/HTML)
    def find_plain_text(parts):
        for part in parts:
            mime_type = part.get('mimeType')
            # Only grab the plain text part
            if mime_type == 'text/plain':
                data = part.get('body', {}).get('data')
                return base64.urlsafe_b64decode(data).decode('utf-8') if data else ""
            # Recursively search in nested parts (common in complex emails)
            if 'parts' in part:
                res = find_plain_text(part['parts'])
                if res: return res
        return ""

    if 'parts' in payload:
        body_text = find_plain_text(payload['parts'])
    else:
        # Fallback for simple emails that aren't multipart
        data = payload.get('body', {}).get('data')
        body_text = base64.urlsafe_b64decode(data).decode('utf-8') if data else ""

    # 4. Return the specific JSON-style dictionary
    return {
        "title": title,
        "sender": sender,
        "date_received": formatted_date,
        "body": body_text.strip()
    }

def get_emails_list(service, days=7):
    query = f"newer_than:{days}d"
    results = service.users().messages().list(userId='me', q=query).execute()
    messages = results.get('messages', [])
    
    cleaned_data = []
    
    for m in messages:
        # Call the transformation function
        simple_msg = get_simplified_message(service, m['id'])
        cleaned_data.append(simple_msg)
        
    return cleaned_data

In [43]:
service = login_gmail(credential_path, token_path)

In [44]:
emails = get_emails_list(service, days=3)

In [46]:
print(json.dumps(emails, indent = 4))

[
    {
        "title": "Transaction Alerts",
        "sender": "\"PayLah! Alerts\" <paylah.alert@dbs.com>",
        "date_received": "2026-03-18 19:46:47",
        "body": ""
    },
    {
        "title": "Job Opportunity as Solution Engineer||KL, Malaysia",
        "sender": "RIDIK Pte Ltd <opportunities@foundit.sg>",
        "date_received": "2026-03-18 19:21:27",
        "body": ""
    },
    {
        "title": "<ADV> Enjoy up to 1.98%*p.a. on your Savings/Current Account Incremental Balances!",
        "sender": "CIMB Singapore <retail@promotion.cimb.com.sg>",
        "date_received": "2026-03-18 19:08:06",
        "body": ""
    },
    {
        "title": "Withholding Tax",
        "sender": "moomoo SG <no_reply@notification.sg.moomoo.com>",
        "date_received": "2026-03-18 15:10:55",
        "body": ""
    },
    {
        "title": "Cash Dividend",
        "sender": "moomoo SG <no_reply@notification.sg.moomoo.com>",
        "date_received": "2026-03-18 15:10:56",
        "bo

In [19]:
results = service.users().messages().list(userId='me', maxResults=10).execute()

In [31]:
messages = results.get('messages', [])
print(json.dumps(messages, indent = 4))


[
    {
        "id": "19d00c510552e4d6",
        "threadId": "19cff2dcc339f2f8"
    },
    {
        "id": "19d00ade4d47d28c",
        "threadId": "19d00ade4d47d28c"
    },
    {
        "id": "19d00a1a85d761b9",
        "threadId": "19d00a1a85d761b9"
    },
    {
        "id": "19cffc8878b9a6ea",
        "threadId": "19cffc8878b9a6ea"
    },
    {
        "id": "19cffc8871278108",
        "threadId": "19cffc8871278108"
    },
    {
        "id": "19cff2dcc339f2f8",
        "threadId": "19cff2dcc339f2f8"
    },
    {
        "id": "19cfe695566d66a1",
        "threadId": "19cfe695566d66a1"
    },
    {
        "id": "19cfc7fe4e89e114",
        "threadId": "19cfc7fe4e89e114"
    },
    {
        "id": "19cfc7fe36d43c4f",
        "threadId": "19cfc7fe36d43c4f"
    },
    {
        "id": "19cfc128b34bac46",
        "threadId": "19cfc128b34bac46"
    }
]


In [34]:
message = service.users().messages().get(userId='me', id=messages[1]['id'], format = 'full').execute()
print(json.dumps(message, indent = 4))

{
    "id": "19d00ade4d47d28c",
    "threadId": "19d00ade4d47d28c",
    "labelIds": [
        "UNREAD",
        "CATEGORY_UPDATES",
        "INBOX"
    ],
    "snippet": "Apply Reply Hello Greetings of the Day, I am sharing the details of the position. We are currently hiring a Solution Engineer for a dynamic role based in KL. Qualifications: 1. Bachelor&#39;s or higher",
    "payload": {
        "partId": "",
        "mimeType": "multipart/mixed",
        "filename": "",
        "headers": [
            {
                "name": "Delivered-To",
                "value": "junming1202@gmail.com"
            },
            {
                "name": "Received",
                "value": "by 2002:a05:7412:35a1:b0:1bb:5459:198f with SMTP id id33csp127416rdb;        Wed, 18 Mar 2026 04:21:30 -0700 (PDT)"
            },
            {
                "name": "X-Received",
                "value": "by 2002:a17:902:f607:b0:2ae:8073:a4ee with SMTP id d9443c01a7336-2b06e38cdfbmr19341285ad.4.17738328

In [ ]:
for message in messages:
    msg = service.users().messages().get(userId='me', id=message['id']).execute()
    print(f"- {msg['snippet']}")

201


In [18]:
service = login_gmail()
print_gmail(service)

Messages:
- Transaction Alerts Problems viewing this email? Select &quot;always display images&quot;. Transaction Ref: IPS77383440512161352 Dear Sir / Madam, We refer to your PayLah! Scan &amp; Pay Transfer dated
- Apply Reply Hello Greetings of the Day, I am sharing the details of the position. We are currently hiring a Solution Engineer for a dynamic role based in KL. Qualifications: 1. Bachelor&#39;s or higher
- From now until 30 April 2026, grow your savings when you top up at least S$10000 in fresh funds into your Savings or Current Account. You will earn bonus interest of up to 1.98% * pa, when combined
- https://moomoo.com/sg Dear Client, UNH paid the dividend of ex-date 2026-03-09, and 14 shares incurred a dividend tax of 9.28 USD. For more details, please check the Funds Details page. Sincerely,
- https://moomoo.com/sg Dear Client, UNH paid the dividend of ex-date 2026-03-09, and 14 shares received 30.94 USD cash dividend. For more details, please check the Funds Details page.